# PS5 QC Inspector: detector training (Colab / Kaggle)

Trains the YOLO detector for one **profile** (`pcb` or `fasteners`) on a free GPU. Then download
`best.pt` and copy it to the Jetson at `~/ps5-qc-inspector/models/vision/<profile>/best.pt`.
The **TensorRT engine is built on the Jetson** (engines are tied to the GPU and TensorRT version).

Runtime > Change runtime type > **T4 GPU** before running.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the project code
Either set `REPO_URL` to your GitHub repo, or leave it empty and upload `training_bundle.zip`
(made on the Jetson with `scripts/make_training_bundle.sh`).

In [ ]:
REPO_URL = ""   # e.g. "https://github.com/<you>/ps5-qc-inspector.git"
PROFILE = "pcb"  # pcb | fasteners

import os, subprocess
os.chdir("/content")
if REPO_URL:
    if not os.path.exists("ps5-qc-inspector"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "ps5-qc-inspector"], check=True)
else:
    from google.colab import files
    up = files.upload()  # choose training_bundle.zip
    subprocess.run(["unzip", "-o", "-q", next(iter(up)), "-d", "ps5-qc-inspector"], check=True)
os.chdir("/content/ps5-qc-inspector")
print(os.listdir("."))

In [ ]:
!pip install -q -r requirements/train.txt

## 2. Dataset
`pcb`: DeepPCB is downloaded and converted automatically (group-level train/val split, official test split,
defect-free templates as negatives and as `test_clean`).
`fasteners`: upload your YOLO-format dataset to `data/datasets/fasteners/` with a `dataset.yaml`.

In [ ]:
if PROFILE == "pcb":
    !python training/prepare_deeppcb.py --download --templates --demo 200
!python training/calibrate_severity.py --profile {PROFILE}

## 3. Train
`yolo11n` is fastest on the Jetson; try `yolo11s` if small-defect recall is weak.

In [ ]:
!python training/train.py --profile {PROFILE} --model yolo11n.pt --epochs 100 --imgsz 640 --batch 32

## 4. Evaluate (Colab GPU numbers; the report must use the Jetson numbers from `evaluate.py --backend tensorrt`)

In [ ]:
!python training/evaluate.py --profile {PROFILE} --backend torch --map

## 5. Export ONNX and download artifacts

In [ ]:
!python training/export.py --profile {PROFILE} --to onnx
!zip -q -r artifacts_{PROFILE}.zip models/vision/{PROFILE} configs/profiles/{PROFILE}.size_thresholds.json bench/results runs/train/{PROFILE}
from google.colab import files
files.download(f"artifacts_{PROFILE}.zip")

## 6. On the Jetson
```bash
cd ~/ps5-qc-inspector && unzip -o ~/Downloads/artifacts_pcb.zip
source .venv/bin/activate
python training/export.py --profile pcb --to engine        # builds best.engine with TensorRT FP16
python training/evaluate.py --profile pcb --backend tensorrt
```